# X2P Rabi 幅度校准

该流程用两个连续 `X2P` 扫描 active XY2 setting 的幅度。实验只生成候选值；确认前不会修改当前配置。

In [ ]:
from uuid import uuid4

from sqvm.calibration import (
    apply_calibration_candidates_to_current_configuration,
    run_rabi,
)

print('SQVM Rabi 用户接口导入成功')

## 扫描参数

设置目标、幅度范围和步进，单位均为 GHz。范围必须从零开始，端点会被包含。其余执行参数使用 API 默认值。

In [ ]:
TARGET = 'Q1'
AMPLITUDE_RANGE_GHZ = (0.0, 0.2)
AMPLITUDE_STEP_GHZ = 0.005
OPERATION_ID = str(uuid4())
RUN_EXPERIMENT = True
UPDATE_CANDIDATE = False

In [ ]:
def report_progress(event):
    action = '开始' if event['event'] == 'circuit_started' else '完成'
    current = event['completed'] + 1 if event['event'] == 'circuit_started' else event['completed']
    print(f"{action} {current}/{event['total']}: {event['circuit_id']}")

if RUN_EXPERIMENT:
    result = run_rabi(
        target=TARGET,
        amplitude_range_GHz=AMPLITUDE_RANGE_GHZ,
        amplitude_step_GHz=AMPLITUDE_STEP_GHZ,
        operation_id=OPERATION_ID,
        progress_callback=report_progress,
    )
    print(f'运行 ID: {result.run_id}')
    print(f'结果目录: {result.root}')
    print(f'amplitude_GHz: {list(result.data["amplitude_GHz"])}')
    print(f'P1: {list(result.data["P1"])}')
    print(f'候选: {result.candidates[TARGET]}')
else:
    print('参数已设置。将 RUN_EXPERIMENT 改为 True 后重新运行本单元格。')

## 查看并确认候选

检查首个 Rabi 峰、拟合质量和候选幅度。确认无误后将 `UPDATE_CANDIDATE` 改为 `True`，再运行下方单元格。

In [ ]:
if UPDATE_CANDIDATE:
    if not RUN_EXPERIMENT or 'result' not in globals():
        raise RuntimeError('请先运行实验并检查候选幅度')
    candidate_id = result.candidates[TARGET]['candidate_id']
    update = apply_calibration_candidates_to_current_configuration(
        result,
        confirmation_phrase=f'APPLY CALIBRATION CANDIDATES {result.run_id}',
        candidate_ids=[candidate_id],
        actor_id='project.manager',
    )
    print(f'已更新当前配置: {update.device_id} r{update.current_revision}')
    print(f'已更新参数: {dict(update.applied_values)}')
else:
    print('候选幅度尚未写入当前配置；确认后将 UPDATE_CANDIDATE 改为 True。')

## Web 查看结果

启动 `start_calibration_web.cmd` 后访问 [http://127.0.0.1:8765/#/experiments](http://127.0.0.1:8765/#/experiments)。Web 控制台只查看结果和确认候选，不启动实验。